In [88]:
import polars as pl
from elfen.extractor import Extractor
from elfen.surface import get_num_tokens_per_sentence
from elfen.pos import get_num_per_pos
from elfen.morphological import get_morph_feats

df = pl.read_csv("/Users/gloria/Documents/uniproject/thesis/data/adress_clean.csv", truncate_ragged_lines=True)
#df = pl.read_csv("/Users/gloria/Documents/uniproject/thesis/data/ccc_clean.csv")

# Keep metadata for adress
df = df.select([
    "ID",
    "age",
    "gender",
    "mmse",
    "diagnosis",
    "aggregated_utterances"
])
extractor = Extractor(data=df, text_column="aggregated_utterances")

print(f"Loaded {len(df)} rows. Columns: {df.columns}")

Loaded 156 rows. Columns: ['ID', 'age', 'gender', 'mmse', 'diagnosis', 'aggregated_utterances']


In [89]:
# ---- FEATURE EXTRACTION ----

# surface / readability
extractor.extract(features=["avg_word_length", "n_polysyllables", "tokens_per_sentence", "n_monosyllables", "n_lemmas", "ttr"])
extractor.data = get_num_tokens_per_sentence(data=extractor.data, backbone="spacy")

# lexical richness
extractor.extract(features=["lexical_density" "lemma_token_ratio"])

# psycholinguistic / semantic
extractor.extract(features=["avg_aoa", "avg_concreteness","n_low_concreteness","n_low_synsets", "n_low_aoa", "avg_sensorimotor", "max_sensorimotor"])

# emotion / sentiment
extractor.extract(features=["avg_valence", "avg_dominance", "sentiment_score"])

# syntax / dependency
extractor.extract(features=["tree_depth"])

# POS
extractor.data = get_num_per_pos(data=extractor.data, backbone="spacy", pos_tags=["NOUN", "VERB", "ADV", "PRON", "INTJ", "DET", "ADJ", "PART", "AUX", "NUM"])

# morphology
morph_config = {"VERB": {"Tense": ["Past", "Pres"]}}
extractor.data = get_morph_feats(data=extractor.data, backbone="spacy", morph_config=morph_config)

print(extractor.data.head())
print([c for c in extractor.data.columns if "Tense" in c or "PronType" in c])

Feature tokens_per_sentence not found. Check spelling.
Feature lexical_densitylemma_token_ratio not found. Check spelling.
Feature max_sensorimotor not found. Check spelling.
shape: (5, 57)
┌──────┬─────┬────────┬──────┬───┬───────┬───────┬───────────────────┬───────────────────┐
│ ID   ┆ age ┆ gender ┆ mmse ┆ … ┆ n_aux ┆ n_num ┆ n_VERB_Tense_Past ┆ n_VERB_Tense_Pres │
│ ---  ┆ --- ┆ ---    ┆ ---  ┆   ┆ ---   ┆ ---   ┆ ---               ┆ ---               │
│ str  ┆ i64 ┆ i64    ┆ str  ┆   ┆ u16   ┆ u16   ┆ u16               ┆ u16               │
╞══════╪═════╪════════╪══════╪═══╪═══════╪═══════╪═══════════════════╪═══════════════════╡
│ S001 ┆ 74  ┆ 0      ┆ NA   ┆ … ┆ 10    ┆ 4     ┆ 2                 ┆ 17                │
│ S002 ┆ 62  ┆ 1      ┆ 30.0 ┆ … ┆ 7     ┆ 0     ┆ 0                 ┆ 8                 │
│ S003 ┆ 69  ┆ 1      ┆ 29.0 ┆ … ┆ 19    ┆ 0     ┆ 1                 ┆ 12                │
│ S004 ┆ 71  ┆ 1      ┆ 30.0 ┆ … ┆ 28    ┆ 0     ┆ 3                 ┆ 23         

In [90]:
# ---- ADJACENT REPETITION (custom) ----

def get_adjacent_repetitions(data: pl.DataFrame) -> pl.DataFrame:
    counts = []
    for doc in data["nlp"].to_list():
        tokens = [tok.lower_ for tok in doc if not tok.is_space]
        count = sum(1 for i in range(len(tokens) - 1) if tokens[i] == tokens[i + 1])
        counts.append(count)
    return data.with_columns(pl.Series("n_adjacent_repetitions", counts, dtype=pl.UInt16))

extractor.data = get_adjacent_repetitions(extractor.data)
print("n_adjacent_repetitions sample:", extractor.data["n_adjacent_repetitions"].head(5).to_list())

n_adjacent_repetitions sample: [3, 0, 0, 3, 0]


In [91]:
# ---- RATIO FEATURES ----
extractor.data = extractor.data.with_columns([
    (pl.col("n_pron") / (pl.col("n_noun") + 1)).alias("pronoun_noun_ratio"),
    (pl.col("n_verb") / (pl.col("n_noun") + 1)).alias("verb_noun_ratio"),
    (pl.col("n_intj") / (pl.col("n_noun") + 1)).alias("filler_to_noun_ratio"),
    (pl.col("n_noun") / (pl.col("n_verb") + 1)).alias("noun_verb_ratio"),
    (pl.col("n_det")  / (pl.col("n_noun") + 1)).alias("det_noun_ratio"),
    (pl.col("n_aux") / (pl.col("n_verb") + 1)).alias("aux_verb_ratio"),
    (pl.col("n_part") / (pl.col("n_verb") + 1)).alias("particle_verb_ratio"),
])

In [92]:
print(len(extractor.data.columns))
print(extractor.data.columns)

65
['ID', 'age', 'gender', 'mmse', 'diagnosis', 'aggregated_utterances', 'nlp', 'n_tokens', 'n_characters', 'avg_word_length', 'n_polysyllables', 'n_monosyllables', 'n_lemmas', 'n_types', 'ttr', 'n_sentences', 'tokens_per_sentence', 'lemmas', 'avg_aoa', 'avg_concreteness', 'n_low_concreteness', 'synsets', 'synsets_noun', 'synsets_verb', 'synsets_adj', 'synsets_adv', 'n_low_synsets', 'n_low_aoa', 'avg_Auditory_sensorimotor', 'avg_Gustatory_sensorimotor', 'avg_Haptic_sensorimotor', 'avg_Interoceptive_sensorimotor', 'avg_Olfactory_sensorimotor', 'avg_Visual_sensorimotor', 'avg_Foot_leg_sensorimotor', 'avg_Hand_arm_sensorimotor', 'avg_Head_sensorimotor', 'avg_Mouth_sensorimotor', 'avg_Torso_sensorimotor', 'avg_valence', 'avg_dominance', 'n_positive_sentiment', 'n_negative_sentiment', 'sentiment_score', 'tree_depth', 'n_noun', 'n_verb', 'n_adv', 'n_pron', 'n_intj', 'n_det', 'n_adj', 'n_part', 'n_aux', 'n_num', 'n_VERB_Tense_Past', 'n_VERB_Tense_Pres', 'n_adjacent_repetitions', 'pronoun_noun

In [93]:
# ---- NORMALIZATION ----

#extractor.token_normalize("all")

extractor.token_normalize([
    "n_polysyllables",
    "n_monosyllables",
    "n_lemmas",
    "n_low_concreteness",
    "n_low_synsets",
    "n_noun",
    "n_verb",
    "n_adv",
    "n_pron",
    "n_intj",
    "n_det",
    "n_adj",
    "n_VERB_Tense_Past",
    "n_VERB_Tense_Pres",
    "n_adjacent_repetitions",
    "n_low_aoa",   
    "n_num",
])

extractor.data = extractor.data.drop([
    "n_tokens",
    "n_characters",
    "n_sentences",
    "n_types",
    "n_lexical_tokens",
    "token_freqs",
    "tokens",
    "lemmas",
    "synsets",
    "synsets_noun",
    "synsets_verb",
    "synsets_adj",
    "synsets_adv",
    "n_positive_sentiment",
    "n_negative_sentiment",
], strict=False)
print(extractor.data.head(5))

shape: (5, 53)
┌──────┬─────┬────────┬──────┬───┬────────────────┬────────────────┬───────────────┬───────────────┐
│ ID   ┆ age ┆ gender ┆ mmse ┆ … ┆ noun_verb_rati ┆ det_noun_ratio ┆ aux_verb_rati ┆ particle_verb │
│ ---  ┆ --- ┆ ---    ┆ ---  ┆   ┆ o              ┆ ---            ┆ o             ┆ _ratio        │
│ str  ┆ i64 ┆ i64    ┆ str  ┆   ┆ ---            ┆ f64            ┆ ---           ┆ ---           │
│      ┆     ┆        ┆      ┆   ┆ f64            ┆                ┆ f64           ┆ f64           │
╞══════╪═════╪════════╪══════╪═══╪════════════════╪════════════════╪═══════════════╪═══════════════╡
│ S001 ┆ 74  ┆ 0      ┆ NA   ┆ … ┆ 1.318182       ┆ 0.733333       ┆ 0.454545      ┆ 0.136364      │
│ S002 ┆ 62  ┆ 1      ┆ 30.0 ┆ … ┆ 1.6            ┆ 0.64           ┆ 0.466667      ┆ 0.2           │
│ S003 ┆ 69  ┆ 1      ┆ 29.0 ┆ … ┆ 1.666667       ┆ 0.692308       ┆ 1.266667      ┆ 0.0           │
│ S004 ┆ 71  ┆ 1      ┆ 30.0 ┆ … ┆ 0.948718       ┆ 0.684211       ┆ 0.71794

In [94]:
col = extractor.data["n_adjacent_repetitions"]
print(col.min(), col.max(), col.dtype)


0.0 0.1016949152542373 Float64


In [95]:
# ---- OUTPUT ----

df_result = extractor.data.drop(["nlp"], strict=False)

print("FINAL SHAPE:", df_result.shape)
print("FINAL COLUMNS:", df_result.columns)

#df_result.write_csv("/Users/gloria/Documents/uniproject/thesis/outputs/ccc_extractedfeatures.csv")
df_result.write_csv("/Users/gloria/Documents/uniproject/thesis/outputs/adress_extractedfeatures.csv")

FINAL SHAPE: (156, 52)
FINAL COLUMNS: ['ID', 'age', 'gender', 'mmse', 'diagnosis', 'aggregated_utterances', 'avg_word_length', 'n_polysyllables', 'n_monosyllables', 'n_lemmas', 'ttr', 'tokens_per_sentence', 'avg_aoa', 'avg_concreteness', 'n_low_concreteness', 'n_low_synsets', 'n_low_aoa', 'avg_Auditory_sensorimotor', 'avg_Gustatory_sensorimotor', 'avg_Haptic_sensorimotor', 'avg_Interoceptive_sensorimotor', 'avg_Olfactory_sensorimotor', 'avg_Visual_sensorimotor', 'avg_Foot_leg_sensorimotor', 'avg_Hand_arm_sensorimotor', 'avg_Head_sensorimotor', 'avg_Mouth_sensorimotor', 'avg_Torso_sensorimotor', 'avg_valence', 'avg_dominance', 'sentiment_score', 'tree_depth', 'n_noun', 'n_verb', 'n_adv', 'n_pron', 'n_intj', 'n_det', 'n_adj', 'n_part', 'n_aux', 'n_num', 'n_VERB_Tense_Past', 'n_VERB_Tense_Pres', 'n_adjacent_repetitions', 'pronoun_noun_ratio', 'verb_noun_ratio', 'filler_to_noun_ratio', 'noun_verb_ratio', 'det_noun_ratio', 'aux_verb_ratio', 'particle_verb_ratio']
